In [0]:
bronze = spark.table("estudo_estrangeiros.bronze.player_standard_stats")
bronze.printSchema()


root
 |-- Rk: string (nullable = true)
 |-- Player: string (nullable = true)
 |-- Nation: string (nullable = true)
 |-- Pos: string (nullable = true)
 |-- Squad: string (nullable = true)
 |-- Age: double (nullable = true)
 |-- Born: double (nullable = true)
 |-- MP: double (nullable = true)
 |-- Starts: double (nullable = true)
 |-- Min: double (nullable = true)
 |-- 90s: double (nullable = true)
 |-- Gls: double (nullable = true)
 |-- Ast: double (nullable = true)
 |-- G+A: double (nullable = true)
 |-- G-PK: double (nullable = true)
 |-- PK: double (nullable = true)
 |-- PKatt: double (nullable = true)
 |-- CrdY: double (nullable = true)
 |-- CrdR: double (nullable = true)
 |-- Gls.1: double (nullable = true)
 |-- Ast.1: double (nullable = true)
 |-- G+A.1: double (nullable = true)
 |-- G-PK.1: double (nullable = true)
 |-- G+A-PK: double (nullable = true)
 |-- Matches: string (nullable = true)
 |-- -9999: string (nullable = true)
 |-- _season: long (nullable = true)
 |-- _source_tab

In [0]:
# Silver - player_season_club (temporada piloto 2014)

import re
from pyspark.sql import functions as F
from pyspark.sql.types import ArrayType, StringType

def to_int(col):
    return F.expr(f"try_cast(try_cast(`{col}` AS DOUBLE) AS INT)")

def split_positions(pos_raw):
    if pos_raw is None:
        return []
    clean = re.sub(r"[^A-Za-z]", "", pos_raw)
    return [clean[i:i+2] for i in range(0, len(clean), 2)]

split_positions_udf = F.udf(split_positions, ArrayType(StringType()))

bronze = spark.table("estudo_estrangeiros.bronze.player_standard_stats")

silver = (
    bronze
    .select(
        F.col("`-9999`").alias("player_id"),
        F.col("Player").alias("player_name"),
        F.col("Nation").alias("nation_raw"),
        F.col("Pos").alias("position_raw"),
        F.col("Squad").alias("squad"),
        to_int("Age").alias("age_fbref"),
        to_int("Born").alias("birth_year"),
        to_int("MP").alias("matches_played"),
        to_int("Starts").alias("starts"),
        to_int("Min").alias("minutes"),
        to_int("Gls").alias("goals"),
        to_int("Ast").alias("assists"),
        to_int("_season").alias("season"),
    )
    .filter(F.col("player_id").isNotNull())
    .withColumn("nation_code", F.trim(F.element_at(F.split("nation_raw", " "), -1)))
    .withColumn("is_brazilian", F.col("nation_code") == "BRA")
    .withColumn("position_parts", split_positions_udf("position_raw"))
    .withColumn("position_primary", F.get(F.col("position_parts"), 0))
    .withColumn("position_secondary", F.get(F.col("position_parts"), 1))
    .drop("position_parts")
    .withColumn("age", F.col("season") - F.col("birth_year"))
    .withColumn(
        "age_band",
        F.when(F.col("age").isNull(), None)
         .when(F.col("age") <= 23, "<=23")
         .when(F.col("age") <= 29, "24-29")
         .otherwise("30+")
    )
    .dropDuplicates(["player_id", "season", "squad"])
)

silver.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable(
    "estudo_estrangeiros.silver.player_season_club"
)
display(silver.limit(20))




player_id,player_name,nation_raw,position_raw,squad,age_fbref,birth_year,matches_played,starts,minutes,goals,assists,season,nation_code,is_brazilian,position_primary,position_secondary,age,age_band
89346085,Marcelo Hermes,br BRA,DFMF,Juventude,29,1995,22,20,1858,2,5,2025,BRA,true,DF,MF,30,30+
054849d2,Iago,br BRA,DFMF,Bahia,27,1997,15,7,626,1,0,2025,BRA,true,DF,MF,28,24-29
37ef00eb,Maykon Jesus,br BRA,DFMF,Vitória,19,2005,9,8,766,0,0,2025,BRA,true,DF,MF,20,<=23
59b55d8c,José,br BRA,MF,Cruzeiro,19,2005,1,0,1,0,0,2025,BRA,true,MF,null,20,<=23
c7495dda,Renato Kayser,br BRA,FW,Vitória,28,1996,30,25,2083,9,0,2025,BRA,true,FW,null,29,24-29
492ade3c,Kenji,br BRA,FW,Cruzeiro,19,2006,3,0,39,0,0,2025,BRA,true,FW,null,19,<=23
1115f8aa,Kevyson,br BRA,MF,Santos,20,2004,1,0,16,0,0,2025,BRA,true,MF,null,21,<=23
467aed9b,Lelê,br BRA,FW,Ceará,27,1997,7,0,110,0,0,2025,BRA,true,FW,null,28,24-29
a80fe10c,Kauan Lindes,br BRA,MF,Botafogo–RJ,21,2003,1,0,31,0,0,2025,BRA,true,MF,null,22,<=23
eddf365e,David Loiola,br BRA,DF,Botafogo–RJ,22,2002,22,20,1830,1,1,2025,BRA,true,DF,null,23,<=23


In [0]:
### Silver - league_table

def to_double(col):
    return F.expr(f"try_cast(`{col}` AS DOUBLE)")

bronze_league = spark.table("estudo_estrangeiros.bronze.league_table")

silver_league_table = (
    bronze_league
    .select(
        to_int("Rk").alias("position"),
        F.col("Squad").alias("squad"),
        to_int("MP").alias("matches_played"),
        to_int("W").alias("wins"),
        to_int("D").alias("draws"),
        to_int("L").alias("losses"),
        to_int("GF").alias("goals_for"),
        to_int("GA").alias("goals_against"),
        to_int("GD").alias("goal_diff"),
        to_int("Pts").alias("points"),
        to_double("Pts/MP").alias("points_per_match"),
        to_int("_season").alias("season"),
    )
)

silver_league_table.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable(
    "estudo_estrangeiros.silver.league_table"
)
display(silver_league_table)


position,squad,matches_played,wins,draws,losses,goals_for,goals_against,goal_diff,points,points_per_match,season
1,Cruzeiro,38,24,8,6,67,38,29,80,2.11,2014
2,São Paulo,38,20,10,8,59,40,19,70,1.84,2014
3,Internacional,38,21,6,11,53,41,12,69,1.82,2014
4,Corinthians,38,19,12,7,49,31,18,69,1.82,2014
5,Atlético Mineiro,38,17,11,10,51,38,13,62,1.63,2014
6,Fluminense,38,17,10,11,61,42,19,61,1.61,2014
7,Grêmio,38,17,10,11,36,24,12,61,1.61,2014
8,Athletico–PR,38,15,9,14,43,42,1,54,1.42,2014
9,Santos,38,15,8,15,42,35,7,53,1.39,2014
10,Flamengo,38,14,10,14,46,47,-1,52,1.37,2014
